# Data Cleaning

### Overview
This document outlines the initial **Data Profiling** and **Data Cleaning** pipeline for all datasets provided by the Organizers. 

**Objectives of this notebook:**
* Review and inspect the overall data structure.
* Standardize data types across all tables.
* Handle duplicate records and missing values (NaNs).

**Goal:** To establish a clean and reliable foundational dataset, ready for the team's downstream tasks, including in-depth **Exploratory Data Analysis (EDA)** and **Machine Learning Modeling**.

## 1. Setup & Data Loading
In this section, we initialize our analytical environment, import required libraries, and ingest the pre-cleaned datasets for downstream processing.

### 1.1. Import Libraries & Environment Setup

In [1]:
import pandas as pd
import os
import warnings

# Ignore warning messages for cleaner notebook output
warnings.filterwarnings('ignore')

### 1.2. Load Datasets
Loading the raw `.csv` files from the dataset.

In [2]:
DATA_PATH = '../dataset/01_raw/'
print("Loading data...")

# Master
df_products = pd.read_csv(DATA_PATH + 'products.csv', low_memory=False)
df_customers   = pd.read_csv(DATA_PATH + 'customers.csv', low_memory=False)
df_promotions  = pd.read_csv(DATA_PATH + 'promotions.csv', low_memory=False)
df_geography   = pd.read_csv(DATA_PATH + 'geography.csv', low_memory=False)

# Transaction
df_orders  = pd.read_csv(DATA_PATH + 'orders.csv', low_memory=False)
df_order_items = pd.read_csv(DATA_PATH + 'order_items.csv', low_memory=False)
df_payments  = pd.read_csv(DATA_PATH + 'payments.csv', low_memory=False)
df_shipments = pd.read_csv(DATA_PATH + 'shipments.csv', low_memory=False)
df_returns  = pd.read_csv(DATA_PATH + 'returns.csv', low_memory=False)
df_reviews  = pd.read_csv(DATA_PATH + 'reviews.csv', low_memory=False)

# Analytical
df_sales = pd.read_csv(DATA_PATH + 'sales.csv', low_memory=False)
# df_sample_submission = pd.read_csv(DATA_PATH + 'sample_submission.csv', low_memory=False)

# Operational
df_inventory = pd.read_csv(DATA_PATH + 'inventory.csv', low_memory=False)
df_web_traffic = pd.read_csv(DATA_PATH + 'web_traffic.csv', low_memory=False)

print("Load data successfully")

Loading data...
Load data successfully


## 2: Data Understanding & Profiling
* **Initial Data Review:** Inspect and evaluate the overall datasets without applying any modifications or manipulations.

#### 2.1 Product categories

In [3]:
print("The overall structure of data:\n")
df_products.info()
print("\nThe random 5 lines of data:")
df_products.sample(5)

The overall structure of data:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2412 entries, 0 to 2411
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   product_id    2412 non-null   int64  
 1   product_name  2412 non-null   object 
 2   category      2412 non-null   object 
 3   segment       2412 non-null   object 
 4   size          2412 non-null   object 
 5   color         2412 non-null   object 
 6   price         2412 non-null   float64
 7   cogs          2412 non-null   float64
dtypes: float64(2), int64(1), object(5)
memory usage: 150.9+ KB

The random 5 lines of data:


,product_id,product_name,category,segment,size,color,price,cogs
1283,2140,PhoenixWear UR-16,Streetwear,Standard,S,red,6611.85000,4492.090890
1260,2033,UrbanVN UR-36,Streetwear,Standard,M,blue,42.27298,26.125297
740,375,LotusWear UE-96,Streetwear,Performance,XL,purple,9362.19375,7022.581532
1481,1121,MekongFit RS-09,Outdoor,Premium,M,black,1007.37000,957.001500
1975,59,DragonWear RP-14,Outdoor,Activewear,XL,yellow,4325.34375,3585.709969


In [4]:
duplicate_data = df_products[df_products.duplicated(keep=False)].sort_values(by=df_products.columns.tolist())
if len(duplicate_data) > 0:
    print(f"{len(duplicate_data)} duplicated rows:")
    display(duplicate_data.head())

duplicate_primary_key = df_products[df_products['product_id'].duplicated(keep=False)].sort_values(by='product_id')
if len(duplicate_primary_key) > 0:
    print(f"{len(duplicate_primary_key)} duplicated product_id rows:")
    display(duplicate_primary_key.head())

print("\nDescriptive statistics of data:")
display(df_products.describe())
display(df_products.describe(include='O'))


Descriptive statistics of data:


,product_id,price,cogs
count,2412.000000,2412.000000,2412.000000
mean,1206.500000,4928.216231,3868.346732
std,696.428747,4776.737669,3878.584151
min,1.000000,9.056594,5.183829
25%,603.750000,59.444924,35.066367
50%,1206.500000,4399.605000,3184.934093
75%,1809.250000,7720.513784,5864.916462
max,2412.000000,40950.000000,38902.500000


,product_name,category,segment,size,color
count,2412,2412,2412,2412,2412
unique,2172,4,8,4,10
top,VietMode RP-02,Streetwear,Activewear,S,orange
freq,3,1320,598,603,242


##### Profiling Summary: 
- **Data Structure:** 2,412 rows and 8 columns.
- **Data Types:** - `product_id`: Currently `int64`. However, since this is a unique identifier with no mathematical significance, it should be converted to `string` for easier analysis.
    - `product_name`: Currently `object`. To facilitate analysis and optimize memory usage, it will be converted to `string`.
    - `category`, `segment`, `size`, and `color`: Currently `object`. Because the number of unique values in these variables is relatively small compared to the total number of rows, they should be cast to `category` to optimize memory and grouping operations.
    - `price` and `cogs`: Currently `float64` → Appropriate.
- **Missing Values:** No missing values detected across the dataset.
- **Duplicate Values:** There are no fully duplicated rows. However, the descriptive statistics reveal only **2,172** unique `product_name` values. This indicates that some items have identical names but different IDs. This requires further investigation to determine if the exact same physical product is being assigned multiple `product_id`s.
- **Descriptive Statistics:** - Both `price` and `cogs` have a **mean > median** → Right-skewed distribution, indicating the presence of some high-priced products.
  - The standard deviation for both variables is **large** (approximately equal to the mean). The data is highly dispersed, showing that products span across a wide variety of price segments.
  - The value range is extremely broad:
    - `price`: from ~9 to ~40,950  
    - `cogs`: from ~5 to ~38,902  
    → Strong likelihood of **outliers** or massive disparities between different product groups.
  → **Conclusion:** The data is logically sound from a business perspective, but it may require further **outlier treatment**. Applying a **log transformation** could also be considered for deeper analysis.

In [5]:
duplicate_primary_key = df_products[df_products['product_name'].duplicated(keep=False)].sort_values(by='product_name')
if len(duplicate_primary_key) > 0:
    print(f"{len(duplicate_primary_key)} duplicated product_name rows:")
    display(duplicate_primary_key.head(10))

426 duplicated product_name rows:


,product_id,product_name,category,segment,size,color,price,cogs
745,380,LotusWear UE-01,Streetwear,Performance,S,red,34.036218,21.343813
645,280,LotusWear UE-01,Streetwear,Performance,S,red,12596.850000,11967.007500
646,281,LotusWear UE-02,Streetwear,Performance,M,black,15746.850000,8750.524545
746,381,LotusWear UE-02,Streetwear,Performance,M,black,21.651582,12.139172
647,282,LotusWear UE-03,Streetwear,Performance,L,orange,14486.850000,12008.149965
747,382,LotusWear UE-03,Streetwear,Performance,L,orange,11147.850000,10315.105605
648,283,LotusWear UE-04,Streetwear,Performance,XL,blue,24.024859,15.410534
748,383,LotusWear UE-04,Streetwear,Performance,XL,blue,7286.850000,6664.553010
649,284,LotusWear UE-05,Streetwear,Performance,S,white,31.046489,18.375704
749,384,LotusWear UE-05,Streetwear,Performance,S,white,12596.850000,10771.566435


##### Duplicate Data Analysis in `product_name`:
- There are 426 rows with duplicate `product_name`s. This is logically sound since a single product can have multiple variants (e.g., different sizes or colors).
- However, closer inspection reveals cases where rows share the exact same `product_name`, `category`, `segment`, `size`, and `color`, yet exhibit massive discrepancies in their selling price (`price`) and cost of goods sold (`cogs`).
- This is a clear indicator of a data quality issue, which could lead to:
  - Distorted revenue and profit analysis.
  - Double counting when performing joins with other tables.
  - Inaccuracies and biases in downstream predictive models.

→ **Action Item:** This issue must be flagged for further investigation and cross-validated once the related tables have been profiled.

#### 2.2 Customers

In [6]:
print("The overall structure of data:\n")
df_customers.info()
print("\nThe random 5 lines of data:")
df_customers.sample(5)

The overall structure of data:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 121930 entries, 0 to 121929
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype 
---  ------               --------------   ----- 
 0   customer_id          121930 non-null  int64 
 1   zip                  121930 non-null  int64 
 2   city                 121930 non-null  object
 3   signup_date          121930 non-null  object
 4   gender               121930 non-null  object
 5   age_group            121930 non-null  object
 6   acquisition_channel  121930 non-null  object
dtypes: int64(2), object(5)
memory usage: 6.5+ MB

The random 5 lines of data:


,customer_id,zip,city,signup_date,gender,age_group,acquisition_channel
45687,58921,1525,Thai Nguyen,2019-03-07,Female,25-34,paid_search
6313,8106,18425,Nam Dinh,2017-11-15,Female,35-44,organic_search
52189,67376,23356,Hanoi,2022-11-15,Male,25-34,organic_search
36781,47401,61931,Bac Giang,2020-07-05,Female,55+,social_media
12356,15952,13338,Cam Pha,2018-04-21,Male,35-44,social_media


In [7]:
duplicate_data = df_customers[df_customers.duplicated(keep=False)].sort_values(by=df_customers.columns.tolist())
if len(duplicate_data) > 0:
    print(f"{len(duplicate_data)} duplicated rows:")
    display(duplicate_data.head())

duplicate_primary_key = df_customers[df_customers['customer_id'].duplicated(keep=False)].sort_values(by='customer_id')
if len(duplicate_primary_key) > 0:
    print(f"{len(duplicate_primary_key)} duplicated customer_id rows:")
    display(duplicate_primary_key.head())

print("\nDescriptive statistics of data:")
display(df_customers.describe())
display(df_customers.describe(include='O'))


Descriptive statistics of data:


,customer_id,zip
count,121930.000000,121930.000000
mean,78736.898663,50990.165595
std,45492.202886,26871.914605
min,1.000000,1001.000000
25%,39343.500000,28689.250000
50%,78784.500000,49835.000000
75%,118156.750000,73488.000000
max,157563.000000,99950.000000


,city,signup_date,gender,age_group,acquisition_channel
count,121930,121930,121930,121930,121930
unique,42,3941,3,5,6
top,Cam Pha,2022-06-02,Female,25-34,organic_search
freq,4398,78,59640,36342,36450


##### Profiling Summary: 
- **Data Structure:** 121,930 rows and 7 columns.
- **Data Types:** - `customer_id` and `zip`: Currently `int64`. However, since these are identifiers/codes with no mathematical significance, they should be converted to `string` for proper analysis.
    - `gender`, `age_group`, `city`, and `acquisition_channel`: Currently `object`. Given the relatively small number of unique values compared to the total number of rows, they should be cast to `category` to optimize memory usage and facilitate grouping operations.
    - `signup_date`: Currently `object`. Since this represents temporal data, it must be converted to `datetime` format to enable time-based analysis.
- **Missing Values:** No missing values detected across the entire dataset.
- **Duplicate Values:** No completely duplicated rows exist in the dataset.

#### 2.3 Promotional programs

In [8]:
print("The overall structure of data:\n")
df_promotions.info()
print("\nThe random 5 lines of data:")
df_promotions.sample(5)

The overall structure of data:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   promo_id             50 non-null     object 
 1   promo_name           50 non-null     object 
 2   promo_type           50 non-null     object 
 3   discount_value       50 non-null     float64
 4   start_date           50 non-null     object 
 5   end_date             50 non-null     object 
 6   applicable_category  10 non-null     object 
 7   promo_channel        50 non-null     object 
 8   stackable_flag       50 non-null     int64  
 9   min_order_value      50 non-null     int64  
dtypes: float64(1), int64(2), object(7)
memory usage: 4.0+ KB

The random 5 lines of data:


,promo_id,promo_name,promo_type,discount_value,start_date,end_date,applicable_category,promo_channel,stackable_flag,min_order_value
22,PROMO-0023,Fall Launch 2017,percentage,10.0,2017-08-30,2017-10-02,NaN,all_channels,1,0
45,PROMO-0046,Rural Special 2021,percentage,15.0,2021-01-30,2021-03-01,Outdoor,in_store,0,0
7,PROMO-0008,Mid-Year Sale 2014,percentage,18.0,2014-06-23,2014-07-22,NaN,social_media,0,0
14,PROMO-0015,Urban Blowout 2015,fixed,50.0,2015-07-30,2015-09-02,Streetwear,online,0,200000
32,PROMO-0033,Fall Launch 2019,percentage,10.0,2019-08-30,2019-10-01,NaN,email,0,50000


In [9]:
duplicate_data = df_promotions[df_promotions.duplicated(keep=False)].sort_values(by=df_promotions.columns.tolist())
if len(duplicate_data) > 0:
    print(f"{len(duplicate_data)} duplicated rows:")
    display(duplicate_data.head())

duplicate_primary_key = df_promotions[df_promotions['promo_id'].duplicated(keep=False)].sort_values(by='promo_id')
if len(duplicate_primary_key) > 0:
    print(f"{len(duplicate_primary_key)} duplicated promo_id rows:")
    display(duplicate_primary_key.head())

print("\nDescriptive statistics of data:")
display(df_promotions.describe())
display(df_promotions.describe(include='O'))


Descriptive statistics of data:


,discount_value,stackable_flag,min_order_value
count,50.000000,50.000000,50.000000
mean,18.500000,0.240000,46000.000000
std,11.241777,0.431419,66116.779802
min,10.000000,0.000000,0.000000
25%,12.000000,0.000000,0.000000
50%,16.500000,0.000000,0.000000
75%,20.000000,0.000000,100000.000000
max,50.000000,1.000000,200000.000000


,promo_id,promo_name,promo_type,start_date,end_date,applicable_category,promo_channel
count,50,50,50,50,50,10,50
unique,50,50,2,50,50,2,5
top,PROMO-0001,Spring Sale 2013,percentage,2013-03-18,2013-04-17,Streetwear,all_channels
freq,1,1,45,1,1,5,19


##### Profiling Summary: 
- **Data Structure:** 50 rows and 10 columns.
- **Data Types:** - `promo_id` and `promo_name`: Currently `object`. To facilitate analysis and optimize memory usage, they will be converted to `string`.
    - `promo_type`, `applicable_category`, and `promo_channel`: Currently `object`. Given the relatively small number of unique values compared to the total number of rows, they should be cast to `category` to streamline grouping and classification.
    - `start_date` and `end_date`: Currently `object`. Since these represent temporal data, they must be converted to `datetime` format to enable time-based analysis.
    - `discount_value`: Currently `float64` → Appropriate, but this variable needs to be split into two independent columns to avoid storing two different metrics (percentage vs. fixed monetary amount) in the same feature.
    - `min_order_value`: Currently `int64` → Appropriate.
    - `stackable_flag`: Currently `int64`. However, with a min of 0 and a max of 1, this is highly likely a categorical/boolean variable → Requires further verification to cast it to a more suitable data type (e.g., `boolean`).
- **Missing Values:** Missing values are present in the `applicable_category` column. According to the data dictionary, missing data here indicates that the promotion applies to all categories. Therefore, we need to fill these NaNs with the value **"All"**.
- **Duplicate Values:** No duplicated rows exist in the dataset.
- **Descriptive Statistics:** - `discount_value`: Since this currently mixes percentages and fixed monetary values, it cannot be meaningfully evaluated using descriptive statistics until it is split.
    - `min_order_value`: The **median = 0** and **Q1 = 0**, indicating that over 50% of the discount campaigns do not require a minimum order value. With **Q3 ≈ 100,000**, only about 25% of campaigns require a substantial minimum threshold. The **max = 200,000** shows that certain campaigns set very high thresholds, likely targeting high-value orders.
    - `min_order_value`: The **mean (~46,000)** is significantly higher than the **median (0)** → highly right-skewed distribution, driven by a few large values. Furthermore, the standard deviation is very large (~66,117), showing high data dispersion with widely varying threshold levels.
    → **Conclusion:** Overall, `min_order_value` is highly uneven, showing a clear polarization between campaigns with no minimum requirement (0) and those with high thresholds. This likely reflects distinct, targeted promotional strategies employed by the business.

In [10]:
print(df_promotions['stackable_flag'].value_counts())

stackable_flag
0    38
1    12
Name: count, dtype: int64


Kết quả cho thấy `stackable_flag` chỉ tồn tại 2 giá trị 0 và 1 -> là một biến phân loại như mong đợi, cần chuyển sang kiểu `bool`

#### 2.4 Geography

In [11]:
print("The overall structure of data:\n")
df_geography.info()
print("\nThe random 5 lines of data:")
df_geography.sample(5)

The overall structure of data:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39948 entries, 0 to 39947
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   zip       39948 non-null  int64 
 1   city      39948 non-null  object
 2   region    39948 non-null  object
 3   district  39948 non-null  object
dtypes: int64(1), object(3)
memory usage: 1.2+ MB

The random 5 lines of data:


,zip,city,region,district
31813,76486,Quang Ngai,Central,District #22
32751,78057,Phan Rang-Thap Cham,Central,District #23
5327,32205,Nam Dinh,East,District #12
15877,29849,Nam Dinh,East,District #11
21556,50569,Quy Nhon,Central,District #30


In [12]:
duplicate_data = df_geography[df_geography.duplicated(keep=False)].sort_values(by=df_geography.columns.tolist())
if len(duplicate_data) > 0:
    print(f"{len(duplicate_data)} duplicated rows:")
    display(duplicate_data.head())

duplicate_primary_key = df_geography[df_geography['zip'].duplicated(keep=False)].sort_values(by='zip')
if len(duplicate_primary_key) > 0:
    print(f"{len(duplicate_primary_key)} duplicated zip rows:")
    display(duplicate_primary_key.head())

print("\nDescriptive statistics of data:")
display(df_geography.describe())
display(df_geography.describe(include='O'))


Descriptive statistics of data:


,zip
count,39948.000000
mean,50895.084735
std,27042.257341
min,1.000000
25%,28279.500000
50%,49876.500000
75%,73526.250000
max,99950.000000


,city,region,district
count,39948,39948,39948
unique,42,3,39
top,Cam Pha,East,District #25
freq,1403,18929,1597


##### Profiling Summary: 
- **Data Structure:** 39,948 rows and 4 columns.
- **Data Types:** - `zip`: Currently `int64`. However, since this is a geographical identifier with no mathematical significance, it should be converted to `string` for proper analysis.
    - `city`, `region`, and `district`: Currently `object`. Given the relatively small number of unique values in these variables compared to the total number of rows, they should be cast to `category` to optimize memory usage and facilitate classification/grouping operations.
- **Missing Values:** No missing values detected across the entire dataset.
- **Duplicate Values:** No completely duplicated rows exist in the dataset.

#### 2.5 Orders

In [13]:
print("The overall structure of data:\n")
df_orders.info()
print("\nThe random 5 lines of data:")
df_orders.sample(5)

The overall structure of data:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 646945 entries, 0 to 646944
Data columns (total 8 columns):
 #   Column          Non-Null Count   Dtype 
---  ------          --------------   ----- 
 0   order_id        646945 non-null  int64 
 1   order_date      646945 non-null  object
 2   customer_id     646945 non-null  int64 
 3   zip             646945 non-null  int64 
 4   order_status    646945 non-null  object
 5   payment_method  646945 non-null  object
 6   device_type     646945 non-null  object
 7   order_source    646945 non-null  object
dtypes: int64(3), object(5)
memory usage: 39.5+ MB

The random 5 lines of data:


,order_id,order_date,customer_id,zip,order_status,payment_method,device_type,order_source
335041,432136,2016-09-11,61795,31322,delivered,apple_pay,mobile,social_media
29487,38100,2012-12-25,6818,17512,cancelled,credit_card,desktop,referral
337631,435450,2016-09-27,130688,78640,delivered,credit_card,tablet,organic_search
512972,661484,2019-04-29,105465,50428,delivered,credit_card,desktop,paid_search
595742,768179,2021-06-27,38701,25696,delivered,cod,desktop,email_campaign


In [14]:
duplicate_data = df_orders[df_orders.duplicated(keep=False)].sort_values(by=df_orders.columns.tolist())
if len(duplicate_data) > 0:
    print(f"{len(duplicate_data)} duplicated rows:")
    display(duplicate_data.head())

duplicate_primary_key = df_orders[df_orders['order_id'].duplicated(keep=False)].sort_values(by='order_id')
if len(duplicate_primary_key) > 0:
    print(f"{len(duplicate_primary_key)} duplicated order_id rows:")
    display(duplicate_primary_key.head())

print("\nDescriptive statistics of data:")
display(df_orders.describe())
display(df_orders.describe(include='O'))


Descriptive statistics of data:


,order_id,customer_id,zip
count,646945.000000,646945.000000,646945.000000
mean,417189.470332,84906.203535,55410.740423
std,240785.704463,48446.922752,28876.471824
min,1.000000,1.000000,1001.000000
25%,208728.000000,41336.000000,30904.000000
50%,417211.000000,87279.000000,54129.000000
75%,625628.000000,133282.000000,83301.000000
max,834397.000000,157563.000000,99950.000000


,order_date,order_status,payment_method,device_type,order_source
count,646945,646945,646945,646945,646945
unique,3833,6,5,3,6
top,2018-05-30,delivered,credit_card,mobile,organic_search
freq,803,516716,356352,291482,181495


##### Profiling Summary: 
- **Data Structure:** 646,945 rows and 8 columns.
- **Data Types:** - `order_id`, `customer_id`, and `zip`: Currently `int64`. However, since these are identifiers with no mathematical significance, they should be converted to `string` for proper analysis.
    - `order_status`, `payment_method`, `device_type`, and `order_source`: Currently `object`. Given the relatively small number of unique values in these variables compared to the total number of rows, they should be cast to `category` to optimize memory usage and facilitate classification.
    - `order_date`: Currently `object`. Since this represents temporal data, it must be converted to `datetime` format to enable time-based analysis.
- **Missing Values:** No missing values detected across the entire dataset.
- **Duplicate Values:** No completely duplicated rows exist in the dataset.

#### 2.6 Order detail items

In [15]:
print("The overall structure of data:\n")
df_order_items.info()
print("\nThe random 5 lines of data:")
df_order_items.sample(5)

The overall structure of data:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 714669 entries, 0 to 714668
Data columns (total 7 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   order_id         714669 non-null  int64  
 1   product_id       714669 non-null  int64  
 2   quantity         714669 non-null  int64  
 3   unit_price       714669 non-null  float64
 4   discount_amount  714669 non-null  float64
 5   promo_id         276316 non-null  object 
 6   promo_id_2       206 non-null     object 
dtypes: float64(2), int64(3), object(2)
memory usage: 38.2+ MB

The random 5 lines of data:


,order_id,product_id,quantity,unit_price,discount_amount,promo_id,promo_id_2
711951,831175,452,4,10635.92,8508.74,PROMO-0050,NaN
48166,54023,716,3,1869.24,0.00,NaN,NaN
16562,19259,1452,7,4446.87,0.00,NaN,NaN
680085,792048,764,3,3937.43,0.00,NaN,NaN
23783,27474,1105,2,3111.11,0.00,NaN,NaN


In [16]:
duplicate_data = df_order_items[df_order_items.duplicated(keep=False)].sort_values(by=df_order_items.columns.tolist())
if len(duplicate_data) > 0:
    print(f"{len(duplicate_data)} duplicated rows:")
    display(duplicate_data.head())

print("\nDescriptive statistics of data:")
display(df_order_items.describe())
display(df_order_items.describe(include='O'))


Descriptive statistics of data:


,order_id,product_id,quantity,unit_price,discount_amount
count,714669.000000,714669.000000,714669.000000,714669.000000,714669.000000
mean,411615.076561,1234.931370,4.495988,5114.690157,1048.887415
std,240480.310686,691.332564,2.290143,3774.817912,2280.530606
min,1.000000,1.000000,1.000000,392.570000,0.000000
25%,203229.000000,689.000000,2.000000,1906.890000,0.000000
50%,409306.000000,990.000000,4.000000,4257.770000,0.000000
75%,618981.000000,2045.000000,6.000000,7273.760000,967.630000
max,834397.000000,2412.000000,8.000000,43056.000000,35235.470000


,promo_id,promo_id_2
count,276316,206
unique,50,2
top,PROMO-0014,PROMO-0015
freq,11451,132


##### Profiling Summary: 
- **Data Structure:** 714,669 rows and 7 columns.
- **Data Types:** - `order_id`, `product_id`, `promo_id`, and `promo_id_2`: Currently `int64` and `object`. However, since these are identifiers with no mathematical significance, they should be converted to `string` to optimize memory and facilitate analysis.
    - `unit_price` and `discount_amount`: Currently `float64` → Appropriate.
    - `quantity`: Currently `int64` → Appropriate.
- **Missing Values:** Missing values are present in the `promo_id` and `promo_id_2` columns. This occurs because no promotions were applied to these order items. Therefore, we need to fill these missing values with **"None"** to indicate the absence of a promotion.
- **Duplicate Values:** No completely duplicated rows exist in the dataset.
- **Descriptive Statistics:** - `unit_price`: **mean > median** → Right-skewed distribution, indicating the presence of some high-priced products. The standard deviation of `unit_price` (**~3,775**) is quite large compared to the mean, showing that selling prices are highly dispersed, reflecting various product segments.
  - Comparing with `price` in the *products.csv* table:
    - Min `unit_price` (~392)  > Min `price` (~9). This suggests that perhaps not all products are sold; only products from the ~390 price segment and above might be generating sales.
    - Max `unit_price` (~43,000)  > Max `price` (~41,000). This strongly suggests that `unit_price` represents dynamic pricing fluctuating around the base `price`.
  - `discount_amount`: **median = 0** and **Q1 = 0**, showing that over 50% of orders do not apply any discount campaigns. With **Q3 ≈ 967**, only about 25% of orders have significant discounts.
  - `discount_amount`: **mean (~1,049) > median (0)** → Highly right-skewed distribution. Additionally, the **max (~35,235)** is very large, suggesting the presence of deep discount campaigns or outliers.
→ **Conclusion:** The data is logically sound from a business perspective, but we may need to further handle **outliers** in `discount_amount` and conduct deeper analysis to determine if `unit_price` is indeed dynamic pricing based on `price`.

#### 2.7 Payments

In [17]:
print("The overall structure of data:\n")
df_payments.info()
print("\nThe random 5 lines of data:")
df_payments.sample(5)

The overall structure of data:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 646945 entries, 0 to 646944
Data columns (total 4 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   order_id        646945 non-null  int64  
 1   payment_method  646945 non-null  object 
 2   payment_value   646945 non-null  float64
 3   installments    646945 non-null  int64  
dtypes: float64(1), int64(2), object(1)
memory usage: 19.7+ MB

The random 5 lines of data:


,order_id,payment_method,payment_value,installments
158974,205164,cod,69340.74,1
211471,272774,credit_card,32148.95,6
129965,167821,credit_card,32136.40,1
531248,685052,credit_card,12374.58,1
508922,656222,cod,7154.84,1


In [18]:
duplicate_data = df_payments[df_payments.duplicated(keep=False)].sort_values(by=df_payments.columns.tolist())
if len(duplicate_data) > 0:
    print(f"{len(duplicate_data)} duplicated rows:")
    display(duplicate_data.head())

duplicate_primary_key = df_payments[df_payments['order_id'].duplicated(keep=False)].sort_values(by='order_id')
if len(duplicate_primary_key) > 0:
    print(f"{len(duplicate_primary_key)} duplicated order_id rows:")
    display(duplicate_primary_key.head())

print("\nDescriptive statistics of data:")
display(df_payments.describe())
display(df_payments.describe(include='O'))


Descriptive statistics of data:


,order_id,payment_value,installments
count,646945.000000,646945.000000,646945.000000
mean,417189.470332,24238.334426,3.448319
std,240785.704463,22378.475324,3.119582
min,1.000000,389.740000,1.000000
25%,208728.000000,7681.060000,1.000000
50%,417211.000000,17229.440000,3.000000
75%,625628.000000,33706.350000,6.000000
max,834397.000000,331570.400000,12.000000


,payment_method
count,646945
unique,5
top,credit_card
freq,356352


##### Profiling Summary: 
- **Data Structure:** 646,945 rows and 4 columns.
- **Data Types:** - `order_id`: Currently `int64`. However, since this is an identifier with no mathematical significance, it should be converted to `string` for proper analysis.
    - `payment_method`: Currently `object`. Given the relatively small number of unique values compared to the total number of rows, it should be cast to `category` to optimize memory usage and facilitate classification.
    - `payment_value`: Currently `float64` → Appropriate.
    - `installments`: Currently `int64`. However, with min = 1 and max = 12, this is highly likely a categorical variable → Requires checking to consider converting to `category`.
- **Missing Values:** No missing values detected across the entire dataset.
- **Duplicate Values:** No completely duplicated rows exist in the dataset.
- **Descriptive Statistics:** - `payment_value`: **mean > median** (~24,238 > ~17,229) → Right-skewed distribution, indicating the presence of high-value orders. Furthermore, the large standard deviation (~22,378) being close to the mean shows strong data dispersion, likely due to the variety in payment values across orders.
    - `payment_value`: Has a wide range with **min ≈ 390** and **max ≈ 331,570**. It is highly likely that outliers exist, as percentiles show that **50%** of orders fall within the **(~7,681 → ~33,706)** range. Additionally, **median < mean** indicates that the vast majority of orders have moderate values, with only a few large orders pulling the average up.
→ **Conclusion:** Overall, `payment_value` exhibits a right-skewed distribution with significant variance. This highlights the diversity of order values and suggests the need to handle **outliers** and consider a **log transformation** for deeper analysis.

In [19]:
print(df_payments['installments'].value_counts())

installments
1     262866
3     218949
6     109910
12     54126
2       1094
Name: count, dtype: int64


The results show that `installments` has only 5 distinct values, confirming it is a categorical variable as expected. It should be converted to the `category` data type.

#### 2.8 Shipments

In [20]:
print("The overall structure of data:\n")
df_shipments.info()
print("\nThe random 5 lines of data:")
df_shipments.sample(5)

The overall structure of data:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 566067 entries, 0 to 566066
Data columns (total 4 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   order_id       566067 non-null  int64  
 1   ship_date      566067 non-null  object 
 2   delivery_date  566067 non-null  object 
 3   shipping_fee   566067 non-null  float64
dtypes: float64(1), int64(1), object(2)
memory usage: 17.3+ MB

The random 5 lines of data:


,order_id,ship_date,delivery_date,shipping_fee
282173,414670,2016-07-17,2016-07-20,0.70
411020,603367,2018-06-16,2018-06-23,2.85
79945,117669,2013-09-22,2013-09-25,0.36
529282,777796,2021-09-05,2021-09-08,1.15
39482,58182,2013-03-27,2013-04-01,25.53


In [21]:
duplicate_data = df_shipments[df_shipments.duplicated(keep=False)].sort_values(by=df_shipments.columns.tolist())
if len(duplicate_data) > 0:
    print(f"{len(duplicate_data)} duplicated rows:")
    display(duplicate_data.head())

duplicate_primary_key = df_shipments[df_shipments['order_id'].duplicated(keep=False)].sort_values(by='order_id')
if len(duplicate_primary_key) > 0:
    print(f"{len(duplicate_primary_key)} duplicated order_id rows:")
    display(duplicate_primary_key.head())

print("\nDescriptive statistics of data:")
display(df_shipments.describe())
display(df_shipments.describe(include='O'))


Descriptive statistics of data:


,order_id,shipping_fee
count,566067.000000,566067.000000
mean,415816.869664,4.962857
std,240007.311562,8.887355
min,1.000000,0.000000
25%,208192.500000,0.870000
50%,415866.000000,1.730000
75%,623218.500000,2.600000
max,834325.000000,32.000000


,ship_date,delivery_date
count,566067,566067
unique,3831,3831
top,2018-06-02,2018-06-06
freq,678,562


##### Profiling Summary: 
- **Data Structure:** 5,566,067 rows and 4 columns.
- **Data Types:** - `order_id`: Currently `int64`. Since this is an identifier with no mathematical significance, it should be converted to `string`.
    - `ship_date` and `delivery_date`: Currently `object`. Since these represent temporal data, they must be converted to `datetime` format.
    - `shipping_fee`: Currently `float64` → Appropriate.
- **Missing Values:** No missing values detected across the entire dataset.
- **Duplicate Values:** No completely duplicated rows exist in the dataset.
- **Descriptive Statistics:** - `shipping_fee`: **mean > median** (~4.96 > ~1.73) → Right-skewed distribution, indicating some orders with high shipping fees. Furthermore, the standard deviation (~8.89) is larger than the mean → data is highly dispersed, indicating potential outliers.
    - `shipping_fee`: Ranges from **min ≈ 0** to **max ≈ 32**. There are orders with free shipping.
    - Percentiles show that **50%** of shipping fees fall within the **(~0.87 → ~2.60)** range. Also, **median < mean** shows that the majority of orders have low shipping fees, with only a few high values pulling the average up.
→ **Conclusion:** Overall, `shipping_fee` is right-skewed due to a few high-fee orders, while most shipping fees are low or free. This likely reflects a popular free-shipping policy combined with specific cases incurring high fees (e.g., express delivery, special orders).

#### 2.9 Returns

In [22]:
print("The overall structure of data:\n")
df_returns.info()
print("\nThe random 5 lines of data:")
df_returns.sample(5)

The overall structure of data:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39939 entries, 0 to 39938
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   return_id        39939 non-null  object 
 1   order_id         39939 non-null  int64  
 2   product_id       39939 non-null  int64  
 3   return_date      39939 non-null  object 
 4   return_reason    39939 non-null  object 
 5   return_quantity  39939 non-null  int64  
 6   refund_amount    39939 non-null  float64
dtypes: float64(1), int64(3), object(3)
memory usage: 2.1+ MB

The random 5 lines of data:


,return_id,order_id,product_id,return_date,return_reason,return_quantity,refund_amount
18705,RET-024061,378077,1920,2016-05-04,not_as_described,4,11626.28
28959,RET-037246,593513,2380,2018-06-19,wrong_size,4,12659.18
29274,RET-037651,600476,2336,2018-07-01,changed_mind,5,42119.83
22419,RET-028816,455864,793,2017-01-10,late_delivery,4,3094.15
19581,RET-025177,395901,1081,2016-06-14,changed_mind,3,8459.51


In [23]:
duplicate_data = df_returns[df_returns.duplicated(keep=False)].sort_values(by=df_returns.columns.tolist())
if len(duplicate_data) > 0:
    print(f"{len(duplicate_data)} duplicated rows:")
    display(duplicate_data.head())

duplicate_primary_key = df_returns[df_returns['return_id'].duplicated(keep=False)].sort_values(by='return_id')
if len(duplicate_primary_key) > 0:
    print(f"{len(duplicate_primary_key)} duplicated return_id rows:")
    display(duplicate_primary_key.head())

print("\nDescriptive statistics of data:")
display(df_returns.describe())
display(df_returns.describe(include='O'))


Descriptive statistics of data:


,order_id,product_id,return_quantity,refund_amount
count,39939.000000,39939.000000,39939.000000,39939.000000
mean,409061.984176,1244.232730,2.743834,12784.458964
std,240063.904576,691.747822,1.828260,14092.150154
min,2.000000,3.000000,1.000000,458.810000
25%,202651.000000,702.000000,1.000000,3573.395000
50%,404254.000000,992.000000,2.000000,7888.880000
75%,615620.000000,2048.000000,4.000000,16881.990000
max,833351.000000,2412.000000,8.000000,160937.940000


,return_id,return_date,return_reason
count,39939,39939,39939
unique,39939,3806,5
top,RET-051504,2016-06-18,wrong_size
freq,1,34,13967


##### Profiling Summary: 
- **Data Structure:** 39,939 rows and 7 columns.
- **Data Types:** - `order_id` and `product_id`: Currently `int64`. Identifiers; should be converted to `string`.
    - `return_id`: Currently `object`. To optimize memory and facilitate analysis, convert to `string`.
    - `return_reason`: Currently `object`. Given the small number of unique values, cast to `category`.
    - `return_date`: Currently `object`. Convert to `datetime`.
    - `refund_amount`: Currently `float64` → Appropriate.
    - `return_quantity`: Currently `int64` → Appropriate.
- **Missing Values:** No missing values detected.
- **Duplicate Values:** No completely duplicated rows exist.
- **Descriptive Statistics:** - `return_quantity`: **mean > median** (~2.74 > 2) → Slightly right-skewed distribution.
    - `return_quantity`: Ranges from **min ≈ 1** to **max ≈ 8**. Most values fall in the **1 → 4** range, indicating that most returns involve a small or moderate quantity.
    - `refund_amount`: **mean > median** (~12,784 > ~7,889) → Clearly right-skewed. The large standard deviation (~14,092) being higher than the mean shows refund values are highly dispersed.
    - `refund_amount`: Wide range from **min ≈ 459** to **max ≈ 160,938**. Outliers likely exist, as **50%** of orders fall in the **(~3,573 → ~16,882)** range. Thus, most return orders have moderate refund values, with a few very large refunds.
→ **Conclusion:** Return quantities are generally small, but refund values can fluctuate significantly. The data is right-skewed, requiring further **outlier** treatment and potential **log transformation**.

#### 2.10 Reviews

In [24]:
print("The overall structure of data:\n")
df_reviews.info()
print("\nThe random 5 lines of data:")
df_reviews.sample(5)

The overall structure of data:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 113551 entries, 0 to 113550
Data columns (total 7 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   review_id     113551 non-null  object
 1   order_id      113551 non-null  int64 
 2   product_id    113551 non-null  int64 
 3   customer_id   113551 non-null  int64 
 4   review_date   113551 non-null  object
 5   rating        113551 non-null  int64 
 6   review_title  113551 non-null  object
dtypes: int64(4), object(3)
memory usage: 6.1+ MB

The random 5 lines of data:


,review_id,order_id,product_id,customer_id,review_date,rating,review_title
714,REV-0000955,5327,1927,72582,2012-08-03,4,Happy with purchase
13686,REV-0017688,97595,1041,3740,2013-08-01,5,Great quality
65887,REV-0085046,472309,965,21634,2017-04-22,1,Not as described
81607,REV-0105356,587228,2066,880,2018-05-28,5,Great quality
833,REV-0001097,6132,796,134884,2012-08-17,5,Highly recommend


In [25]:
duplicate_data = df_reviews[df_reviews.duplicated(keep=False)].sort_values(by=df_reviews.columns.tolist())
if len(duplicate_data) > 0:
    print(f"{len(duplicate_data)} duplicated rows:")
    display(duplicate_data.head())

duplicate_primary_key = df_reviews[df_reviews['review_id'].duplicated(keep=False)].sort_values(by='review_id')
if len(duplicate_primary_key) > 0:
    print(f"{len(duplicate_primary_key)} duplicated review_id rows:")
    display(duplicate_primary_key.head())

print("\nDescriptive statistics of data:")
display(df_reviews.describe())
display(df_reviews.describe(include='O'))


Descriptive statistics of data:


,order_id,product_id,customer_id,rating
count,113551.000000,113551.000000,113551.000000,113551.000000
mean,408999.519740,1232.018705,85694.342762,3.936011
std,239021.922809,690.839232,48501.480918,1.149867
min,1.000000,3.000000,2.000000,1.000000
25%,202048.500000,689.000000,42096.000000,3.000000
50%,406841.000000,981.000000,89755.000000,4.000000
75%,614844.000000,2045.000000,133850.000000,5.000000
max,833296.000000,2412.000000,157563.000000,5.000000


,review_id,review_date,review_title
count,113551,113551,113551
unique,113551,3825,18
top,REV-0146984,2017-05-06,Very satisfied
freq,1,77,11450


##### Profiling Summary: 
- **Data Structure:** 113,551 rows and 7 columns.
- **Data Types:** - `order_id`, `product_id`, and `customer_id`: Currently `int64`. Identifiers; should be converted to `string`.
    - `review_id`: Currently `object`. Convert to `string` for memory optimization.
    - `review_title`: Currently `object`. Small number of unique values; convert to `category`.
    - `review_date`: Currently `object`. Convert to `datetime`.
    - `rating`: Currently `int64`. However, with min = 1 and max = 5, this is highly likely a categorical variable → Requires checking to consider converting to `category`.
- **Missing Values:** No missing values detected.
- **Duplicate Values:** No completely duplicated rows exist.

In [26]:
print(df_reviews['rating'].value_counts())

rating
5    45256
4    36412
3    17016
2     9095
1     5772
Name: count, dtype: int64


Kết quả cho thấy `rating` chỉ tồn tại 5 giá trị distinct -> là một biến phân loại như mong đợi, cần chuyển sang kiểu `category`

#### 2.11 Sale daily revenue data

In [27]:
print("The overall structure of data:\n")
df_sales.info()
print("\nThe random 5 lines of data:")
df_sales.sample(5)

The overall structure of data:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3833 entries, 0 to 3832
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   Date     3833 non-null   object 
 1   Revenue  3833 non-null   float64
 2   COGS     3833 non-null   float64
dtypes: float64(2), object(1)
memory usage: 90.0+ KB

The random 5 lines of data:


,Date,Revenue,COGS
244,2013-03-05,2440470.72,1954056.55
1859,2017-08-06,2056024.63,2936220.92
462,2013-10-09,3524600.98,2822967.50
1586,2016-11-06,3177194.93,2614531.12
56,2012-08-29,9260558.32,7336294.54


In [28]:
duplicate_data = df_sales[df_sales.duplicated(keep=False)].sort_values(by=df_sales.columns.tolist())
if len(duplicate_data) > 0:
    print(f"{len(duplicate_data)} duplicated rows:")
    display(duplicate_data.head())

duplicate_primary_key = df_sales[df_sales['Date'].duplicated(keep=False)].sort_values(by='Date')
if len(duplicate_primary_key) > 0:
    print(f"{len(duplicate_primary_key)} duplicated Date rows:")
    display(duplicate_primary_key.head())

print("\nDescriptive statistics of data:")
display(df_sales.describe())
display(df_sales.describe(include='O'))


Descriptive statistics of data:


,Revenue,COGS
count,3.833000e+03,3.833000e+03
mean,4.286584e+06,3.695134e+06
std,2.624840e+06,2.219789e+06
min,2.798139e+05,2.365763e+05
25%,2.471089e+06,2.150580e+06
50%,3.647304e+06,3.161113e+06
75%,5.350877e+06,4.637294e+06
max,2.090527e+07,1.653586e+07


,Date
count,3833
unique,3833
top,2022-12-31
freq,1


##### Profiling Summary: 
- **Data Structure:** 3,833 rows and 3 columns.
- **Data Types:** - `Date`: Currently `object`. Represents temporal data; convert to `datetime`.
    - `Revenue` and `COGS`: Currently `float64` → Appropriate.
- **Missing Values:** No missing values detected.
- **Duplicate Values:** No completely duplicated rows exist.

#### 2.12 Inventory

In [29]:
print("The overall structure of data:\n")
df_inventory.info()
print("\nThe random 5 lines of data:")
df_inventory.sample(5)

The overall structure of data:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60247 entries, 0 to 60246
Data columns (total 17 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   snapshot_date      60247 non-null  object 
 1   product_id         60247 non-null  int64  
 2   stock_on_hand      60247 non-null  int64  
 3   units_received     60247 non-null  int64  
 4   units_sold         60247 non-null  int64  
 5   stockout_days      60247 non-null  int64  
 6   days_of_supply     60247 non-null  float64
 7   fill_rate          60247 non-null  float64
 8   stockout_flag      60247 non-null  int64  
 9   overstock_flag     60247 non-null  int64  
 10  reorder_flag       60247 non-null  int64  
 11  sell_through_rate  60247 non-null  float64
 12  product_name       60247 non-null  object 
 13  category           60247 non-null  object 
 14  segment            60247 non-null  object 
 15  year               60247 non-null  int

,snapshot_date,product_id,stock_on_hand,units_received,units_sold,stockout_days,days_of_supply,fill_rate,stockout_flag,overstock_flag,reorder_flag,sell_through_rate,product_name,category,segment,year,month
55593,2020-06-30,2263,309,6,5,2,1854.0,0.9333,1,1,0,0.0159,VietMotion RP-60,Outdoor,Activewear,2020,6
8591,2017-01-31,539,48,11,9,1,160.0,0.9667,1,1,0,0.1579,SaigonFlex UC-04,Streetwear,Everyday,2017,1
7003,2016-06-30,490,72,27,22,1,98.2,0.9667,1,1,0,0.2340,SaigonFlex UM-95,Streetwear,Balanced,2016,6
31048,2017-04-30,1260,11,1,1,1,330.0,0.9667,1,1,0,0.0833,VietMode MP-28,Casual,Activewear,2017,4
13246,2013-04-30,691,93,21,18,1,155.0,0.9667,1,1,0,0.1622,SaigonFlex UC-56,Streetwear,Everyday,2013,4


In [30]:
duplicate_data = df_inventory[df_inventory.duplicated(keep=False)].sort_values(by=df_inventory.columns.tolist())
if len(duplicate_data) > 0:
    print(f"{len(duplicate_data)} duplicated rows:")
    display(duplicate_data.head())

print("\nDescriptive statistics of data:")
display(df_inventory.describe())
display(df_inventory.describe(include='O'))


Descriptive statistics of data:


,product_id,stock_on_hand,units_received,units_sold,stockout_days,days_of_supply,fill_rate,stockout_flag,overstock_flag,reorder_flag,sell_through_rate,year,month
count,60247.000000,60247.000000,60247.000000,60247.000000,60247.000000,60247.000000,60247.000000,60247.000000,60247.000000,60247.0,60247.000000,60247.000000,60247.000000
mean,1311.408468,189.298455,18.046807,15.417764,1.160639,912.677576,0.961312,0.673411,0.762561,0.0,0.152275,2017.222799,6.617292
std,673.051769,316.976124,34.080228,28.404379,1.624490,2587.624108,0.054156,0.468969,0.425517,0.0,0.139291,2.972353,3.385629
min,1.000000,3.000000,1.000000,1.000000,0.000000,5.200000,0.066700,0.000000,0.000000,0.0,0.000400,2012.000000,1.000000
25%,760.000000,15.000000,2.000000,2.000000,0.000000,96.000000,0.933300,0.000000,1.000000,0.0,0.042100,2015.000000,4.000000
50%,1223.000000,62.000000,6.000000,6.000000,1.000000,240.000000,0.966700,1.000000,1.000000,0.0,0.111100,2017.000000,7.000000
75%,1942.000000,210.000000,19.000000,16.000000,2.000000,683.100000,1.000000,1.000000,1.000000,0.0,0.238100,2020.000000,10.000000
max,2412.000000,2673.000000,817.000000,670.000000,28.000000,68100.000000,1.000000,1.000000,1.000000,0.0,0.853100,2022.000000,12.000000


,snapshot_date,product_name,category,segment
count,60247,60247,60247,60247
unique,126,1465,4,8
top,2018-05-31,VietMode RP-10,Streetwear,Activewear
freq,562,203,31020,18290


##### Profiling Summary: 
- **Data Structure:** 60,247 rows and 17 columns.
- **Data Types:** - `product_id`: Currently `int64`. Identifier; should be converted to `string`.
    - `product_name`: Currently `object`. Convert to `string` to optimize memory.
    - `category` and `segment`: Currently `object`. Small number of unique values; convert to `category`.
    - `snapshot_date`: Currently `object`. Convert to `datetime`.
    - `days_of_supply`, `fill_rate`, and `sell_through_rate`: Currently `float64` → Appropriate.
    - `stock_on_hand`, `units_received`, `units_sold`, `stockout_days`, `month`, and `year`: Currently `int64` → Appropriate.
    - `stockout_flag`, `overstock_flag`, and `reorder_flag`: Currently `int64`. With min = 0 and max = 1 (or 0), these are highly likely categorical/boolean variables → Requires checking to cast to a more suitable data type.
- **Missing Values:** No missing values detected.
- **Duplicate Values:** No completely duplicated rows exist.

In [31]:
print(df_inventory['stockout_flag'].value_counts())
print(df_inventory['overstock_flag'].value_counts())
print(df_inventory['reorder_flag'].value_counts())

stockout_flag
1    40571
0    19676
Name: count, dtype: int64
overstock_flag
1    45942
0    14305
Name: count, dtype: int64
reorder_flag
0    60247
Name: count, dtype: int64


The results show that `stackable_flag` (or `stockout_flag`, `overstock_flag`, `reorder_flag`) only contains 2 values (0 and 1) → As expected, these are categorical/boolean variables and should be converted to `bool`.

#### 2.13 Web traffic

In [32]:
print("The overall structure of data:\n")
df_web_traffic.info()
print("\nThe random 5 lines of data:")
df_web_traffic.sample(5)

The overall structure of data:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3652 entries, 0 to 3651
Data columns (total 7 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   date                      3652 non-null   object 
 1   sessions                  3652 non-null   int64  
 2   unique_visitors           3652 non-null   int64  
 3   page_views                3652 non-null   int64  
 4   bounce_rate               3652 non-null   float64
 5   avg_session_duration_sec  3652 non-null   float64
 6   traffic_source            3652 non-null   object 
dtypes: float64(2), int64(3), object(2)
memory usage: 199.8+ KB

The random 5 lines of data:


,date,sessions,unique_visitors,page_views,bounce_rate,avg_session_duration_sec,traffic_source
419,2014-02-24,16007,11852,56518,0.00507,193.7,organic_search
851,2015-05-02,33476,26822,182726,0.00510,219.9,organic_search
2340,2019-05-30,35571,27779,145299,0.00510,276.5,email_campaign
2374,2019-07-03,30229,24504,151719,0.00504,159.1,organic_search
3472,2022-07-05,32386,26366,104994,0.00516,111.2,organic_search


In [33]:
duplicate_data = df_web_traffic[df_web_traffic.duplicated(keep=False)].sort_values(by=df_web_traffic.columns.tolist())
if len(duplicate_data) > 0:
    print(f"{len(duplicate_data)} duplicated rows:")
    display(duplicate_data.head())

duplicate_primary_key = df_web_traffic[df_web_traffic['date'].duplicated(keep=False)].sort_values(by='date')
if len(duplicate_primary_key) > 0:
    print(f"{len(duplicate_primary_key)} duplicated date rows:")
    display(duplicate_primary_key.head())

print("\nDescriptive statistics of data:")
display(df_web_traffic.describe())
display(df_web_traffic.describe(include='O'))


Descriptive statistics of data:


,sessions,unique_visitors,page_views,bounce_rate,avg_session_duration_sec
count,3652.000000,3652.000000,3652.000000,3652.000000,3652.000000
mean,25041.768072,19031.404436,108615.224535,0.004487,210.283242
std,9422.609335,7237.953062,44472.055524,0.000753,63.771711
min,7973.000000,6136.000000,30451.000000,0.003200,100.100000
25%,17099.250000,12915.000000,72982.000000,0.003848,156.700000
50%,23633.500000,17924.000000,101010.500000,0.004450,209.200000
75%,31782.750000,24191.750000,138086.000000,0.005160,266.200000
max,50947.000000,40430.000000,275560.000000,0.005800,319.900000


,date,traffic_source
count,3652,3652
unique,3652,6
top,2022-12-31,organic_search
freq,1,1090


##### Profiling Summary: 
- **Data Structure:** 3,652 rows and 7 columns.
- **Data Types:** - `traffic_source`: Currently `object`. Convert to `category`.
    - `date`: Currently `object`. Convert to `datetime`.
    - `bounce_rate` and `avg_session_duration_sec`: Currently `float64` → Appropriate.
    - `sessions`, `unique_visitors`, and `page_views`: Currently `int64` → Appropriate.
- **Missing Values:** No missing values detected.
- **Duplicate Values:** No completely duplicated rows exist.

## 3: Data Cleaning
Initial data processing and transformation based on the observations and profiling from Section 2.

### 3.1 Standardizing Data Types:

#### 3.1.1 Product categories

In [34]:
cols_to_string = ['product_id', 'product_name']
cols_to_category = ['category', 'segment', 'size', 'color']

df_products[cols_to_string] = df_products[cols_to_string].astype('string')
df_products[cols_to_category] = df_products[cols_to_category].astype('category')

print("Display structure of data after data type casting: \n")
df_products.info()

Display structure of data after data type casting: 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2412 entries, 0 to 2411
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype   
---  ------        --------------  -----   
 0   product_id    2412 non-null   string  
 1   product_name  2412 non-null   string  
 2   category      2412 non-null   category
 3   segment       2412 non-null   category
 4   size          2412 non-null   category
 5   color         2412 non-null   category
 6   price         2412 non-null   float64 
 7   cogs          2412 non-null   float64 
dtypes: category(4), float64(2), string(2)
memory usage: 86.1 KB


The data types of the variables are currently appropriate. However, we need to inspect the unique values in these variables to ensure there are no redundant, overlapping, or meaningless categories.

In [35]:
for val in cols_to_category:
    print(f"Total unique values of '{val}' column: {df_products[val].nunique()}")
    counts = df_products[val].value_counts()
    percents = df_products[val].value_counts(normalize=True) * 100
    summary_df = pd.DataFrame({
        'Count': counts,
        'Percentage': percents.apply(lambda x: f"{x:.2f}%".replace('.', ','))
    })
    print(summary_df.to_string())
    print("-" * 40, "\n")

Total unique values of 'category' column: 4
            Count Percentage
category                    
Streetwear   1320     54,73%
Outdoor       743     30,80%
Casual        201      8,33%
GenZ          148      6,14%
---------------------------------------- 

Total unique values of 'segment' column: 8
             Count Percentage
segment                      
Activewear     598     24,79%
Everyday       405     16,79%
Performance    347     14,39%
Balanced       306     12,69%
Standard       262     10,86%
Premium        177      7,34%
All-weather    169      7,01%
Trendy         148      6,14%
---------------------------------------- 

Total unique values of 'size' column: 4
      Count Percentage
size                  
L       603     25,00%
M       603     25,00%
S       603     25,00%
XL      603     25,00%
---------------------------------------- 

Total unique values of 'color' column: 10
        Count Percentage
color                   
black     242     10,03%
orange    242  

Currently, the unique values ​​in the variables all appear reasonable and meaningful. No further adjustments are needed.

#### 3.1.2 Customers

In [36]:
cols_to_string = ['customer_id', 'zip']
cols_to_category = ['gender', 'age_group', 'city', 'acquisition_channel']
cols_to_datetime = ['signup_date']

df_customers[cols_to_string] = df_customers[cols_to_string].astype('string')
df_customers[cols_to_category] = df_customers[cols_to_category].astype('category')

for col in cols_to_datetime:
    df_customers[col] = pd.to_datetime(df_customers[col], format='%Y-%m-%d', errors='coerce')

print("Display structure of data after data type casting: \n")
df_customers.info()

Display structure of data after data type casting: 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 121930 entries, 0 to 121929
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   customer_id          121930 non-null  string        
 1   zip                  121930 non-null  string        
 2   city                 121930 non-null  category      
 3   signup_date          121930 non-null  datetime64[ns]
 4   gender               121930 non-null  category      
 5   age_group            121930 non-null  category      
 6   acquisition_channel  121930 non-null  category      
dtypes: category(4), datetime64[ns](1), string(2)
memory usage: 3.3 MB


The data types of the variables are currently appropriate. However, we need to inspect the unique values in these variables to ensure there are no redundant, overlapping, or meaningless categories.

In [37]:
for val in cols_to_category:
    print(f"Total unique values of '{val}' column: {df_customers[val].nunique()}")
    counts = df_customers[val].value_counts()
    percents = df_customers[val].value_counts(normalize=True) * 100
    summary_df = pd.DataFrame({
        'Count': counts,
        'Percentage': percents.apply(lambda x: f"{x:.2f}%".replace('.', ','))
    })
    print(summary_df.to_string())
    print("-" * 40, "\n")

Total unique values of 'gender' column: 3
            Count Percentage
gender                      
Female      59640     48,91%
Male        57457     47,12%
Non-binary   4833      3,96%
---------------------------------------- 

Total unique values of 'age_group' column: 5
           Count Percentage
age_group                  
25-34      36342     29,81%
35-44      31920     26,18%
45-54      23172     19,00%
18-24      17039     13,97%
55+        13457     11,04%
---------------------------------------- 

Total unique values of 'city' column: 42
                     Count Percentage
city                                 
Cam Pha               4398      3,61%
Thai Nguyen           4347      3,57%
Phu Ly                4243      3,48%
Hanoi                 4240      3,48%
Ha Long               4236      3,47%
Bac Ninh              4172      3,42%
Hai Phong             4170      3,42%
Nam Dinh              4169      3,42%
Bac Giang             4160      3,41%
Ninh Binh             4081 

Currently, the unique values ​​in the variables all appear reasonable and meaningful. No further adjustments are needed.

#### 3.1.3 Promotional programs

In [38]:
cols_to_string = ['promo_id', 'promo_name']
cols_to_category = ['promo_type', 'applicable_category', 'promo_channel']
cols_to_bool = ['stackable_flag']
cols_to_datetime = ['start_date', 'end_date']

df_promotions[cols_to_string] = df_promotions[cols_to_string].astype('string')
df_promotions[cols_to_category] = df_promotions[cols_to_category].astype('category')
df_promotions[cols_to_bool] = df_promotions[cols_to_bool].astype(bool)

for col in cols_to_datetime:
    df_promotions[col] = pd.to_datetime(df_promotions[col], format='%Y-%m-%d', errors='coerce')

print("Display structure of data after data type casting: \n")
df_promotions.info()

Display structure of data after data type casting: 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   promo_id             50 non-null     string        
 1   promo_name           50 non-null     string        
 2   promo_type           50 non-null     category      
 3   discount_value       50 non-null     float64       
 4   start_date           50 non-null     datetime64[ns]
 5   end_date             50 non-null     datetime64[ns]
 6   applicable_category  10 non-null     category      
 7   promo_channel        50 non-null     category      
 8   stackable_flag       50 non-null     bool          
 9   min_order_value      50 non-null     int64         
dtypes: bool(1), category(3), datetime64[ns](2), float64(1), int64(1), string(2)
memory usage: 3.1 KB


The data types of the variables are currently appropriate. However, we need to inspect the unique values in these variables to ensure there are no redundant, overlapping, or meaningless categories.

In [39]:
for val in cols_to_category:
    print(f"Total unique values of '{val}' column: {df_promotions[val].nunique()}")
    counts = df_promotions[val].value_counts()
    percents = df_promotions[val].value_counts(normalize=True) * 100
    summary_df = pd.DataFrame({
        'Count': counts,
        'Percentage': percents.apply(lambda x: f"{x:.2f}%".replace('.', ','))
    })
    print(summary_df.to_string())
    print("-" * 40, "\n")

Total unique values of 'promo_type' column: 2
            Count Percentage
promo_type                  
percentage     45     90,00%
fixed           5     10,00%
---------------------------------------- 

Total unique values of 'applicable_category' column: 2
                     Count Percentage
applicable_category                  
Outdoor                  5     50,00%
Streetwear               5     50,00%
---------------------------------------- 

Total unique values of 'promo_channel' column: 5
               Count Percentage
promo_channel                  
all_channels      19     38,00%
online            13     26,00%
email              7     14,00%
social_media       6     12,00%
in_store           5     10,00%
---------------------------------------- 



Currently, the unique values ​​in the variables all appear reasonable and meaningful. No further adjustments are needed.

#### 3.1.4 Geography

In [40]:
cols_to_string = ['zip']
cols_to_category = ['city', 'region', 'district']

df_geography[cols_to_string] = df_geography[cols_to_string].astype('string')
df_geography[cols_to_category] = df_geography[cols_to_category].astype('category')

print("Display structure of data after data type casting: \n")
df_geography.info()

Display structure of data after data type casting: 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39948 entries, 0 to 39947
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype   
---  ------    --------------  -----   
 0   zip       39948 non-null  string  
 1   city      39948 non-null  category
 2   region    39948 non-null  category
 3   district  39948 non-null  category
dtypes: category(3), string(1)
memory usage: 432.1 KB


The data types of the variables are currently appropriate. However, we need to inspect the unique values in these variables to ensure there are no redundant, overlapping, or meaningless categories.

In [41]:
for val in cols_to_category:
    print(f"Total unique values of '{val}' column: {df_geography[val].nunique()}")
    counts = df_geography[val].value_counts()
    percents = df_geography[val].value_counts(normalize=True) * 100
    summary_df = pd.DataFrame({
        'Count': counts,
        'Percentage': percents.apply(lambda x: f"{x:.2f}%".replace('.', ','))
    })
    print(summary_df.to_string())
    print("-" * 40, "\n")

Total unique values of 'city' column: 42
                     Count Percentage
city                                 
Cam Pha               1403      3,51%
Phu Ly                1399      3,50%
Thai Nguyen           1394      3,49%
Hanoi                 1376      3,44%
Nam Dinh              1370      3,43%
Ha Long               1357      3,40%
Bac Giang             1347      3,37%
Bac Ninh              1346      3,37%
Hai Phong             1346      3,37%
Son Tay               1344      3,36%
Ninh Binh             1326      3,32%
Uong Bi               1325      3,32%
Viet Tri              1324      3,31%
Lao Cai               1272      3,18%
Kon Tum               1265      3,17%
Hoi An                1255      3,14%
Dong Hoi              1246      3,12%
Phan Rang-Thap Cham   1219      3,05%
Hue                   1216      3,04%
Tuy Hoa               1212      3,03%
Nha Trang             1199      3,00%
Quang Ngai            1192      2,98%
Phan Thiet            1189      2,98%
Tam Ky   

Currently, the unique values ​​in the variables all appear reasonable and meaningful. No further adjustments are needed.

#### 3.1.5 Orders

In [42]:
cols_to_string = ['order_id', 'customer_id', 'zip']
cols_to_category = ['order_status', 'payment_method', 'device_type', 'order_source']
cols_to_datetime = ['order_date']

df_orders[cols_to_string] = df_orders[cols_to_string].astype('string')
df_orders[cols_to_category] = df_orders[cols_to_category].astype('category')

for col in cols_to_datetime:
    df_orders[col] = pd.to_datetime(df_orders[col], format='%Y-%m-%d', errors='coerce')

print("Display structure of data after data type casting: \n")
df_orders.info()

Display structure of data after data type casting: 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 646945 entries, 0 to 646944
Data columns (total 8 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   order_id        646945 non-null  string        
 1   order_date      646945 non-null  datetime64[ns]
 2   customer_id     646945 non-null  string        
 3   zip             646945 non-null  string        
 4   order_status    646945 non-null  category      
 5   payment_method  646945 non-null  category      
 6   device_type     646945 non-null  category      
 7   order_source    646945 non-null  category      
dtypes: category(4), datetime64[ns](1), string(3)
memory usage: 22.2 MB


The data types of the variables are currently appropriate. However, we need to inspect the unique values in these variables to ensure there are no redundant, overlapping, or meaningless categories.

In [43]:
for val in cols_to_category:
    print(f"Total unique values of '{val}' column: {df_orders[val].nunique()}")
    counts = df_orders[val].value_counts()
    percents = df_orders[val].value_counts(normalize=True) * 100
    summary_df = pd.DataFrame({
        'Count': counts,
        'Percentage': percents.apply(lambda x: f"{x:.2f}%".replace('.', ','))
    })
    print(summary_df.to_string())
    print("-" * 40, "\n")

Total unique values of 'order_status' column: 6
               Count Percentage
order_status                   
delivered     516716     79,87%
cancelled      59462      9,19%
returned       36142      5,59%
shipped        13773      2,13%
paid           13577      2,10%
created         7275      1,12%
---------------------------------------- 

Total unique values of 'payment_method' column: 5
                 Count Percentage
payment_method                   
credit_card     356352     55,08%
paypal           97018     15,00%
cod              96681     14,94%
apple_pay        64763     10,01%
bank_transfer    32131      4,97%
---------------------------------------- 

Total unique values of 'device_type' column: 3
              Count Percentage
device_type                   
mobile       291482     45,06%
desktop      258855     40,01%
tablet        96608     14,93%
---------------------------------------- 

Total unique values of 'order_source' column: 6
                 Count Percen

Currently, the unique values ​​in the variables all appear reasonable and meaningful. No further adjustments are needed.

#### 3.1.6 Order detail items

In [44]:
cols_to_string = ['order_id', 'product_id', 'promo_id', 'promo_id_2']

df_order_items[cols_to_string] = df_order_items[cols_to_string].astype('string')

print("Display structure of data after data type casting: \n")
df_order_items.info()

Display structure of data after data type casting: 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 714669 entries, 0 to 714668
Data columns (total 7 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   order_id         714669 non-null  string 
 1   product_id       714669 non-null  string 
 2   quantity         714669 non-null  int64  
 3   unit_price       714669 non-null  float64
 4   discount_amount  714669 non-null  float64
 5   promo_id         276316 non-null  string 
 6   promo_id_2       206 non-null     string 
dtypes: float64(2), int64(1), string(4)
memory usage: 38.2 MB


Currently, the unique values ​​in the variables all appear reasonable and meaningful. No further adjustments are needed.

#### 3.1.7 Payments

In [45]:
df_payments['installments'] = df_payments['installments'].apply(lambda x: f"{x} period" if x == 1 else f"{x} periods")

cols_to_string = ['order_id']
cols_to_category = ['payment_method', 'installments']

df_payments[cols_to_string] = df_payments[cols_to_string].astype('string')
df_payments[cols_to_category] = df_payments[cols_to_category].astype('category')

print("Display structure of data after data type casting: \n")
df_payments.info()

Display structure of data after data type casting: 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 646945 entries, 0 to 646944
Data columns (total 4 columns):
 #   Column          Non-Null Count   Dtype   
---  ------          --------------   -----   
 0   order_id        646945 non-null  string  
 1   payment_method  646945 non-null  category
 2   payment_value   646945 non-null  float64 
 3   installments    646945 non-null  category
dtypes: category(2), float64(1), string(1)
memory usage: 11.1 MB


The data types of the variables are currently appropriate. However, we need to inspect the unique values in these variables to ensure there are no redundant, overlapping, or meaningless categories.

In [46]:
for val in cols_to_category:
    print(f"Total unique values of '{val}' column: {df_payments[val].nunique()}")
    counts = df_payments[val].value_counts()
    percents = df_payments[val].value_counts(normalize=True) * 100
    summary_df = pd.DataFrame({
        'Count': counts,
        'Percentage': percents.apply(lambda x: f"{x:.2f}%".replace('.', ','))
    })
    print(summary_df.to_string())
    print("-" * 40, "\n")

Total unique values of 'payment_method' column: 5
                 Count Percentage
payment_method                   
credit_card     356352     55,08%
paypal           97018     15,00%
cod              96681     14,94%
apple_pay        64763     10,01%
bank_transfer    32131      4,97%
---------------------------------------- 

Total unique values of 'installments' column: 5
               Count Percentage
installments                   
1 period      262866     40,63%
3 periods     218949     33,84%
6 periods     109910     16,99%
12 periods     54126      8,37%
2 periods       1094      0,17%
---------------------------------------- 



Currently, the unique values ​​in the variables all appear reasonable and meaningful. No further adjustments are needed.

#### 3.1.8 Shipments

In [47]:
cols_to_string = ['order_id']
cols_to_datetime = ['ship_date', 'delivery_date']

df_shipments[cols_to_string] = df_shipments[cols_to_string].astype('string')

for col in cols_to_datetime:
    df_shipments[col] = pd.to_datetime(df_shipments[col], format='%Y-%m-%d', errors='coerce')

print("Display structure of data after data type casting: \n")
df_shipments.info()

Display structure of data after data type casting: 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 566067 entries, 0 to 566066
Data columns (total 4 columns):
 #   Column         Non-Null Count   Dtype         
---  ------         --------------   -----         
 0   order_id       566067 non-null  string        
 1   ship_date      566067 non-null  datetime64[ns]
 2   delivery_date  566067 non-null  datetime64[ns]
 3   shipping_fee   566067 non-null  float64       
dtypes: datetime64[ns](2), float64(1), string(1)
memory usage: 17.3 MB


Currently, the unique values ​​in the variables all appear reasonable and meaningful. No further adjustments are needed.

#### 3.1.9 Returns

In [48]:
cols_to_string = ['order_id', 'product_id', 'return_id']
cols_to_category = ['return_reason']
cols_to_datetime = ['return_date']

df_returns[cols_to_string] = df_returns[cols_to_string].astype('string')
df_returns[cols_to_category] = df_returns[cols_to_category].astype('category')

for col in cols_to_datetime:
    df_returns[col] = pd.to_datetime(df_returns[col], format='%Y-%m-%d', errors='coerce')

print("Display structure of data after data type casting: \n")
df_returns.info()

Display structure of data after data type casting: 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39939 entries, 0 to 39938
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   return_id        39939 non-null  string        
 1   order_id         39939 non-null  string        
 2   product_id       39939 non-null  string        
 3   return_date      39939 non-null  datetime64[ns]
 4   return_reason    39939 non-null  category      
 5   return_quantity  39939 non-null  int64         
 6   refund_amount    39939 non-null  float64       
dtypes: category(1), datetime64[ns](1), float64(1), int64(1), string(3)
memory usage: 1.9 MB


The data types of the variables are currently appropriate. However, we need to inspect the unique values in these variables to ensure there are no redundant, overlapping, or meaningless categories.

In [49]:
for val in cols_to_category:
    print(f"Total unique values of '{val}' column: {df_returns[val].nunique()}")
    counts = df_returns[val].value_counts()
    percents = df_returns[val].value_counts(normalize=True) * 100
    summary_df = pd.DataFrame({
        'Count': counts,
        'Percentage': percents.apply(lambda x: f"{x:.2f}%".replace('.', ','))
    })
    print(summary_df.to_string())
    print("-" * 40, "\n")

Total unique values of 'return_reason' column: 5
                  Count Percentage
return_reason                     
wrong_size        13967     34,97%
defective          8020     20,08%
not_as_described   7035     17,61%
changed_mind       6931     17,35%
late_delivery      3986      9,98%
---------------------------------------- 



Currently, the unique values ​​in the variables all appear reasonable and meaningful. No further adjustments are needed.

#### 3.1.10 Reviews

In [50]:
df_reviews['rating'] = df_reviews['rating'].apply(lambda x: f"{x}*")

cols_to_string = ['order_id', 'product_id', 'customer_id', 'review_id']
cols_to_category = ['review_title', 'rating']
cols_to_datetime = ['review_date']

df_reviews[cols_to_string] = df_reviews[cols_to_string].astype('string')
df_reviews[cols_to_category] = df_reviews[cols_to_category].astype('category')

for col in cols_to_datetime:
    df_reviews[col] = pd.to_datetime(df_reviews[col], format='%Y-%m-%d', errors='coerce')

print("Display structure of data after data type casting: \n")
df_reviews.info()

Display structure of data after data type casting: 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 113551 entries, 0 to 113550
Data columns (total 7 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   review_id     113551 non-null  string        
 1   order_id      113551 non-null  string        
 2   product_id    113551 non-null  string        
 3   customer_id   113551 non-null  string        
 4   review_date   113551 non-null  datetime64[ns]
 5   rating        113551 non-null  category      
 6   review_title  113551 non-null  category      
dtypes: category(2), datetime64[ns](1), string(4)
memory usage: 4.5 MB


The data types of the variables are currently appropriate. However, we need to inspect the unique values in these variables to ensure there are no redundant, overlapping, or meaningless categories.

In [51]:
for val in cols_to_category:
    print(f"Total unique values of '{val}' column: {df_reviews[val].nunique()}")
    counts = df_reviews[val].value_counts()
    percents = df_reviews[val].value_counts(normalize=True) * 100
    summary_df = pd.DataFrame({
        'Count': counts,
        'Percentage': percents.apply(lambda x: f"{x:.2f}%".replace('.', ','))
    })
    print(summary_df.to_string())
    print("-" * 40, "\n")

Total unique values of 'review_title' column: 18
                         Count Percentage
review_title                             
Very satisfied           11450     10,08%
Highly recommend         11407     10,05%
Great quality            11218      9,88%
Excellent product!       11181      9,85%
Good overall              9185      8,09%
Happy with purchase       9171      8,08%
Solid choice              9070      7,99%
Works well                8986      7,91%
Mixed feelings            5706      5,03%
Average product           5656      4,98%
Decent, nothing special   5654      4,98%
Some issues               3037      2,67%
Would not reorder         3034      2,67%
Below expectations        3024      2,66%
Would not recommend       1460      1,29%
Poor quality              1443      1,27%
Very disappointed         1442      1,27%
Not as described          1427      1,26%
---------------------------------------- 

Total unique values of 'rating' column: 5
        Count Percentage
r

Currently, the unique values ​​in the variables all appear reasonable and meaningful. No further adjustments are needed.

#### 3.1.11 Sale daily revenue data

In [52]:
cols_to_datetime = ['Date']

for col in cols_to_datetime:
    df_sales[col] = pd.to_datetime(df_sales[col], format='%Y-%m-%d', errors='coerce')

print("Display structure of data after data type casting: \n")
df_sales.info()

Display structure of data after data type casting: 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3833 entries, 0 to 3832
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype         
---  ------   --------------  -----         
 0   Date     3833 non-null   datetime64[ns]
 1   Revenue  3833 non-null   float64       
 2   COGS     3833 non-null   float64       
dtypes: datetime64[ns](1), float64(2)
memory usage: 90.0 KB


Currently, the unique values ​​in the variables all appear reasonable and meaningful. No further adjustments are needed.

#### 3.1.12 Inventory

In [53]:
cols_to_string = ['product_id', 'product_name']
cols_to_category = ['category', 'segment']
cols_to_bool = ['stockout_flag', 'overstock_flag', 'reorder_flag']
cols_to_datetime = ['snapshot_date']

df_inventory[cols_to_string] = df_inventory[cols_to_string].astype('string')
df_inventory[cols_to_category] = df_inventory[cols_to_category].astype('category')
df_inventory[cols_to_bool] = df_inventory[cols_to_bool].astype(bool)

for col in cols_to_datetime:
    df_inventory[col] = pd.to_datetime(df_inventory[col], format='%Y-%m-%d', errors='coerce')

print("Display structure of data after data type casting: \n")
df_inventory.info()

Display structure of data after data type casting: 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60247 entries, 0 to 60246
Data columns (total 17 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   snapshot_date      60247 non-null  datetime64[ns]
 1   product_id         60247 non-null  string        
 2   stock_on_hand      60247 non-null  int64         
 3   units_received     60247 non-null  int64         
 4   units_sold         60247 non-null  int64         
 5   stockout_days      60247 non-null  int64         
 6   days_of_supply     60247 non-null  float64       
 7   fill_rate          60247 non-null  float64       
 8   stockout_flag      60247 non-null  bool          
 9   overstock_flag     60247 non-null  bool          
 10  reorder_flag       60247 non-null  bool          
 11  sell_through_rate  60247 non-null  float64       
 12  product_name       60247 non-null  string        
 13  category

The data types of the variables are currently appropriate. However, we need to inspect the unique values in these variables to ensure there are no redundant, overlapping, or meaningless categories.

In [54]:
for val in cols_to_category:
    print(f"Total unique values of '{val}' column: {df_inventory[val].nunique()}")
    counts = df_inventory[val].value_counts()
    percents = df_inventory[val].value_counts(normalize=True) * 100
    summary_df = pd.DataFrame({
        'Count': counts,
        'Percentage': percents.apply(lambda x: f"{x:.2f}%".replace('.', ','))
    })
    print(summary_df.to_string())
    print("-" * 40, "\n")

Total unique values of 'category' column: 4
            Count Percentage
category                    
Streetwear  31020     51,49%
Outdoor     21050     34,94%
GenZ         4674      7,76%
Casual       3503      5,81%
---------------------------------------- 

Total unique values of 'segment' column: 8
             Count Percentage
segment                      
Activewear   18290     30,36%
Everyday     13598     22,57%
Performance   7673     12,74%
Balanced      6622     10,99%
Trendy        4674      7,76%
Premium       3182      5,28%
Standard      3127      5,19%
All-weather   3081      5,11%
---------------------------------------- 



Currently, the unique values ​​in the variables all appear reasonable and meaningful. No further adjustments are needed.

#### 3.1.13 Web traffic

In [55]:
cols_to_category = ['traffic_source']
cols_to_datetime = ['date']

df_web_traffic[cols_to_category] = df_web_traffic[cols_to_category].astype('category')

for col in cols_to_datetime:
    df_web_traffic[col] = pd.to_datetime(df_web_traffic[col], format='%Y-%m-%d', errors='coerce')

print("Display structure of data after data type casting: \n")
df_web_traffic.info()

Display structure of data after data type casting: 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3652 entries, 0 to 3651
Data columns (total 7 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   date                      3652 non-null   datetime64[ns]
 1   sessions                  3652 non-null   int64         
 2   unique_visitors           3652 non-null   int64         
 3   page_views                3652 non-null   int64         
 4   bounce_rate               3652 non-null   float64       
 5   avg_session_duration_sec  3652 non-null   float64       
 6   traffic_source            3652 non-null   category      
dtypes: category(1), datetime64[ns](1), float64(2), int64(3)
memory usage: 175.1 KB


The data types of the variables are currently appropriate. However, we need to inspect the unique values in these variables to ensure there are no redundant, overlapping, or meaningless categories.

In [56]:
for val in cols_to_category:
    print(f"Total unique values of '{val}' column: {df_web_traffic[val].nunique()}")
    counts = df_web_traffic[val].value_counts()
    percents = df_web_traffic[val].value_counts(normalize=True) * 100
    summary_df = pd.DataFrame({
        'Count': counts,
        'Percentage': percents.apply(lambda x: f"{x:.2f}%".replace('.', ','))
    })
    print(summary_df.to_string())
    print("-" * 40, "\n")

Total unique values of 'traffic_source' column: 6
                Count Percentage
traffic_source                  
organic_search   1090     29,85%
paid_search       784     21,47%
social_media      632     17,31%
email_campaign    505     13,83%
referral          375     10,27%
direct            266      7,28%
---------------------------------------- 



Currently, the unique values ​​in the variables all appear reasonable and meaningful. No further adjustments are needed.

### 3.2 Handling Missing Values

#### 3.2.1 Promotional programs

In [57]:
df_promotions['applicable_category'] = df_promotions['applicable_category'].astype('string').fillna('All').astype('category')
df_promotions.info()

print("-" * 40, "\n")
print(f"Total unique values of 'applicable_category' column: {df_promotions['applicable_category'].nunique()}")
counts = df_promotions['applicable_category'].value_counts()
percents = df_promotions['applicable_category'].value_counts(normalize=True) * 100
summary_df = pd.DataFrame({
    'Count': counts,
    'Percentage': percents.apply(lambda x: f"{x:.2f}%".replace('.', ','))
})
print(summary_df.to_string())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   promo_id             50 non-null     string        
 1   promo_name           50 non-null     string        
 2   promo_type           50 non-null     category      
 3   discount_value       50 non-null     float64       
 4   start_date           50 non-null     datetime64[ns]
 5   end_date             50 non-null     datetime64[ns]
 6   applicable_category  50 non-null     category      
 7   promo_channel        50 non-null     category      
 8   stackable_flag       50 non-null     bool          
 9   min_order_value      50 non-null     int64         
dtypes: bool(1), category(3), datetime64[ns](2), float64(1), int64(1), string(2)
memory usage: 3.1 KB
---------------------------------------- 

Total unique values of 'applicable_category' column: 3
             

#### 3.2.2 Order detail items

In [58]:
col_to_check = ['promo_id', 'promo_id_2']
df_order_items[col_to_check] = df_order_items[col_to_check].fillna('None')
df_order_items.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 714669 entries, 0 to 714668
Data columns (total 7 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   order_id         714669 non-null  string 
 1   product_id       714669 non-null  string 
 2   quantity         714669 non-null  int64  
 3   unit_price       714669 non-null  float64
 4   discount_amount  714669 non-null  float64
 5   promo_id         714669 non-null  string 
 6   promo_id_2       714669 non-null  string 
dtypes: float64(2), int64(1), string(4)
memory usage: 38.2 MB


## 4: Data Integration & Export

In this document, all raw datasets provided by the Organizers have gone through Data Profiling, data type standardization, and basic missing/duplicate value treatments. Before saving this as the Cleaned Dataset, we will perform a structural streamlining step.

### 4.1. Data Integration
Through entity-relationship analysis, we observed that the *orders.csv* and *payments.csv* tables share a 1-to-1 relationship (each order has exactly one corresponding payment record). To optimize storage space and avoid repetitive `merge` operations in downstream EDA notebooks, these two tables will be integrated into a single dataset.

In [59]:
df_payments = df_payments.drop(columns=['payment_method'])
df_orders = df_orders.merge(df_payments, on='order_id', how='left')
df_orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 646945 entries, 0 to 646944
Data columns (total 10 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   order_id        646945 non-null  string        
 1   order_date      646945 non-null  datetime64[ns]
 2   customer_id     646945 non-null  string        
 3   zip             646945 non-null  string        
 4   order_status    646945 non-null  category      
 5   payment_method  646945 non-null  category      
 6   device_type     646945 non-null  category      
 7   order_source    646945 non-null  category      
 8   payment_value   646945 non-null  float64       
 9   installments    646945 non-null  category      
dtypes: category(5), datetime64[ns](1), float64(1), string(3)
memory usage: 27.8 MB


### 4.2. Xuất dữ liệu (Export Data)
All cleaned datasets will be exported to the *../dataset/02_after_clean/* directory in *.parquet* format. 

**Why Parquet?** It heavily compresses the file size while strictly preserving the standardized data types (such as `datetime`, `category`, and `string`) defined in Phase 2. This ensures that team members can directly plot charts immediately upon loading the data without re-casting types.

In [60]:
EXPORT_PATH = '../dataset/02_after_clean/'
os.makedirs(EXPORT_PATH, exist_ok=True)

print("Exporting cleaned data...")

# Master
df_products.to_parquet(EXPORT_PATH + 'products.parquet', engine='pyarrow', index=False)
df_customers.to_parquet(EXPORT_PATH + 'customers.parquet', index=False)
df_promotions.to_parquet(EXPORT_PATH + 'promotions.parquet', index=False)
df_geography.to_parquet(EXPORT_PATH + 'geography.parquet', index=False)

# Transaction
df_orders.to_parquet(EXPORT_PATH + 'orders.parquet', index=False)
df_order_items.to_parquet(EXPORT_PATH + 'order_items.parquet', index=False)
df_shipments.to_parquet(EXPORT_PATH + 'shipments.parquet', index=False)
df_returns.to_parquet(EXPORT_PATH + 'returns.parquet', index=False)
df_reviews.to_parquet(EXPORT_PATH + 'reviews.parquet', index=False)

# Analytical
df_sales.to_parquet(EXPORT_PATH + 'sales.parquet', index=False)
#df_sample_submission.to_parquet(EXPORT_PATH + 'sample_submission.parquet', index=False)

# Operational
df_inventory.to_parquet(EXPORT_PATH + 'inventory.parquet', index=False)
df_web_traffic.to_parquet(EXPORT_PATH + 'web_traffic.parquet', index=False)

print("Export data successfully")

Exporting cleaned data...
Export data successfully


### 4.3. Instructions for importing data for subsequent notebooks.
In subsequent stages, when loading data for visualization or modeling, use the Pandas `read_parquet` function pointing to the *02_after_clean* directory.

Example:
```python
import pandas as pd
# Load the cleaned customers dataset
df_customers = pd.read_parquet('../dataset/02_after_clean/customers.parquet')